# Занятие 4. Методы квантовой химии

In [ ]:
# 📥 Установка пакетов и библиотек
! pip install matplotlib
! pip install numpy
! pip install basis_set_exchange

In [ ]:
# 🛠️ Загрузка необходимых пакетов в среду выполнения
import numpy as np
import matplotlib.pyplot as plt
import os

# 🔎 Введение

Теоретическая химия - это лесенка моделей: на каждой следующей ступени мы выбираем более грубую модель, описывающую поведение вещества с большими допущениями, но за это получаем возможность описывать более крупные системы и в большем промежутке времени.

| Уровень | Размер системы | Характерные времена | Программы |
|---|---|---|---|
| **Квантовая механика (QM)** | 1–10³ атомов | фс–пс | ORCA, Gaussian, VASP |
| **Молекулярная механика (MM)** | 10³–10⁶ атомов | нс–мкс | GROMACS, AMBER |
| **Крупнозернистые модели (coarse-grained, CG)** | 10⁵–10⁹ частиц | мкс–мс | MARTINI |
| **Континуальные методы (FEM)** | макрообъекты | любые | COMSOL, ANSYS |

# 🔬 Глава 1. Введение

## 1.0. Уравнение Шрёдингера 

Вся квантовая химия строится на решении уравнения Шрёдингера 🐱, в стационарном виде оно может быть записано как:

$$ \hat{H}Ψ(r,R)=EΨ(r,R) $$

* $\Psi$ — волновая функция системы;
* $\hat{H}$ — гамильтониан, оператор полной энергии;
* $E$ — полная энергия;

Гамильтониан включает пять слагаемых:

$$
\hat{H} =
-\frac{1}{2}\sum_{k}\nabla_{k}^{2}
-\frac{1}{2}\sum_{N}\frac{\nabla_{N}^{2}}{m_{N}}
-\sum_{N,k}\frac{Z_{N}}{r_{Nk}}
+\sum_{N'>N}\frac{Z_{N}Z_{N'}}{r_{NN'}}
+\sum_{j>k}\frac{1}{r_{jk}}
$$

Точное аналитическое решение существует только для одноэлектронных систем (H, He⁺, H₂⁺…) из-за последнего слагаемого - межэлектронного отталкивания.

## 1.1. Приближение Борна-Оппенгеймера

Протон во много раз тяжелее электронов ($m_p/m_e \approx 1836$), поэтому **в большинстве случаев** можно использовать приближение, что электроны успевают мгновенно подстроиться под любое движение ядер. Это позволяет разделить задачу: **зафиксировать ядра** и решать уравнение Шрёдингера только для электронов:

$$ \hat{H}_{el}\,\psi_{el}(\mathbf{r};\, \mathbf{R}) = E_{el}(\mathbf{R})\,\psi_{el}(\mathbf{r};\, \mathbf{R}) $$

Координаты ядер $\mathbf{R}$ входят в уравнение уже не как переменные, а как **параметры** системы. 

## 1.2 Метод Хартри-Фока-Рутана

**Идея Хартри и Фока (ХФ):** пусть каждый электрон движется не в поле конкретных соседей, а в **усреднённом поле** всех остальных электронов (приближение среднего поля).

У электронов есть два важных свойства: (1) они **неразличимы** - нельзя сказать, «какой именно» электрон сидит на какой орбитали, - и (2) они **фермионы**: волновая функция обязана менять знак при перестановке любых двух электронов. Поэтому нужно просуммировать все способы рассадить электроны по орбиталям, чередуя знаки. Ровно так раскрывается определитель матрицы в линейной алгебре - поэтому такую волновую функцию можно записать как **детерминант матрицы Слейтера** (в которой строки - электроны, столбцы - спин-орбитали):

$$
\Psi \approx \frac{1}{\sqrt{N!}}
\begin{vmatrix}
\chi_{1}(1) & \chi_{2}(1) & \cdots & \chi_{N}(1) \\
\chi_{1}(2) & \chi_{2}(2) & \cdots & \chi_{N}(2) \\
\vdots & \vdots & \ddots & \vdots \\
\chi_{1}(N) & \chi_{2}(N) & \cdots & \chi_{N}(N)
\end{vmatrix}
$$

Тогда определитель матрицы Слейтера двух электронов:

$$ \Psi(1,2) = \frac{1}{\sqrt{2}}\Big[\chi_{1}(1)\,\chi_{2}(2) - \chi_{2}(1)\,\chi_{1}(2)\Big] $$

Свойства определителя дают всю нужную физику:
* переставили два электрона = переставили две строки → определитель меняет знак - **антисимметрия**;
* посадили два электрона на одну спин-орбиталь → получили два одинаковых столбца → определитель тождественно равен нулю → система не определена - выполняется **принцип Паули**;

Для примера, два электрона на одной пространственной орбитали $\psi_1$ с разными проекциями спина:

$$
\Psi(1,2) = \frac{1}{\sqrt{2}}
\begin{vmatrix}
\psi_1(\mathbf{r}_1)\,\alpha(1) & \psi_1(\mathbf{r}_1)\,\beta(1) \\
\psi_1(\mathbf{r}_2)\,\alpha(2) & \psi_1(\mathbf{r}_2)\,\beta(2)
\end{vmatrix}
= \frac{1}{\sqrt{2}}\Big[\psi_1(\mathbf{r}_1)\alpha(1)\,\psi_1(\mathbf{r}_2)\beta(2) - \psi_1(\mathbf{r}_1)\beta(1)\,\psi_1(\mathbf{r}_2)\alpha(2)\Big]
$$

Пространственный множитель $\psi_1(\mathbf{r}_1)\psi_1(\mathbf{r}_2)$ общий, его выносим за скобку:

$$ \Psi(1,2) = \psi_{1}(\mathbf{r}_1)\,\psi_{1}(\mathbf{r}_2) \cdot \frac{1}{\sqrt{2}}\Big[\alpha(1)\beta(2) - \beta(1)\alpha(2)\Big] $$

Энергия системы по методу Хартри-Фока, описываемой детерминантом Слейтера может быть рассчитана:

$$ E_{\text{ХФ}} = \langle\Psi|\hat{H}_{el}|\Psi\rangle = \sum_{i} h_{ii} + \frac{1}{2}\sum_{i,j}\big(J_{ij} - K_{ij}\big) $$

* $h_{ii}$ - **одноэлектронная часть**: кинетическая энергия электронов $\chi_i$ и электрон-ядерное взаимодействие;
* $J_{ij}$ - **кулоновский интеграл**: классическое отталкивание двух электронных облаков
* $K_{ij}$ - **обменный интеграл**: появляется из перекрёстных членов определителя и отличен от нуля только для электронов с одинаковой проекцией спина.

Минимизируя $E_{\text{ХФ}}$ изменяя "форму" орбиталей (при условии, что орбитали остаются ортонормированными) - можно получить **уравнение Хартри-Фока**:

$$ \hat{f}\,\chi_{i} = \varepsilon_{i}\,\chi_{i}, \qquad \hat{f} = -\frac{1}{2}\nabla^{2} - \sum_{N}\frac{Z_{N}}{r_{N}} + \sum_{j}\big(\hat{J}_{j} - \hat{K}_{j}\big) $$


### 1.2.1 Метод самосогласованного поля (SCF)

Проблема оптимизации "формы" орбиталей для поиска минимальной энергии $E_{\text{ХФ}}$ заключается в том, что оператор Фока $\hat{f}$ включает операторы $\hat{J}$ и $\hat{K}$ для расчёта энергии в среднем поле, но чтобы получить это среднее поле - нам также нужны орбитали, а их (орбитали) мы как раз и оптимизируем, минимизируя энергию. Поэтому метод самосогласованного поля итерационный: 

1. угадываем стартовые орбитали;
2. строим из них оператор Фока $\hat{f}$;
3. решаем уравнения Фока
4. энергия стала меньше? Да - возвращаемся к шагу 2 и берём новые орбитали. Нет - готово.

Идея Рутана - разложить каждую орбиталь по набору заранее выбранных известных функций $\varphi_k$ - базисному набору с некими коэффициентами $c_{ki}$:

$$ \chi_{i} = \sum_{k} c_{ki}\,\varphi_{k} $$

Тогда задача поиска орбиталей нужной формы сводится к оптимизации коэффициентов **$c_{ki}$**. Подставив разложение орбиталей по базисным функциям в уравнение Фока, получаем **уравнение Рутана-Холла**:

$$
\sum_{k} \langle \varphi_{\mu}|\hat{f}|\varphi_{k}\rangle\, c_{ki}
\;=\;
\varepsilon_{i} \sum_{k} \langle \varphi_{\mu}|\varphi_{k}\rangle\, c_{ki}
$$


$$ \mathbf{F}\,\mathbf{C} = \mathbf{S}\,\mathbf{C}\,\boldsymbol{\varepsilon} $$

* $\mathbf{F}$ - матрица Фока, в которой среднее поле строится из занятых орбиталей, поэтому $\mathbf{F}$ **зависит от $\mathbf{C}$**;
* $\mathbf{S}$ - матрица перекрывания: $S_{\mu k} = \langle \varphi_\mu|\varphi_k\rangle$.
* $\mathbf{C}$ - матрица искомых коэффициентов $c_{ki}$.
* $\boldsymbol{\varepsilon}$ - диагональная матрица орбитальных энергий.

### 1.2.2 Ограниченный и неограниченный метод Хартри-Фока

**Ограниченный метод Хартри-Фока (RHF)** предполагает, что $\alpha$- и $\beta$-электроны обязаны делить общие пространственные орбитали. Для подавляющего большинства стабильных молекул в синглетном состоянии это работает. Но бывают системы с неспаренными электронами. Тогда мы можем использовать два подхода:

* **Неограниченный метод (UHF).** У $\alpha$- и $\beta$-электронов свои, независимые наборы пространственных орбиталей: $\psi_i^{\alpha} \neq \psi_i^{\beta}$. Вариационной свободы больше, поэтому энергия получается ниже (или равна) RHF, но детерминант перестаёт быть чистым спиновым состоянием.

* **Ограниченный метод для открытых оболочек (ROHF).** Компромисс: спаренные электроны делят общие орбитали, неспаренные получают собственные. Спинового загрязнения нет по построению, но гибкости меньше, а сходимость капризнее.

## 1.3 Теория функционала плотности 

В методе Хартри-Фока мы упрощали электрон-электронное взаимодействие, но переменная оставалась прежней - волновая функция, зависящая от $4N$ параметров ($3N$ координат и $N$ спинов). А **электронная плотность** $\rho(\mathbf{r})$ - функция всего трёх переменных (координат), сколько бы электронов ни было в системе. Любая система может характеризоваться распределением электронной плотности, а можно ли оценивать энергию системы прямо из плотности, минуя волновую функцию?

### 1.3.1 Теоремы Хоэнберга и Кона

**Теорема 1 (существование).** 

Электронная плотность основного состояния однозначно определяет молекулу (положения и заряды ядер), а значит - гамильтониан и все свойства системы. Точный функционал $E[\rho]$ существует, причём его «межэлектронная» часть универсальна.

**Теорема 2 (вариационный принцип).** Для любой пробной электронной плотности $\tilde{\rho}$ (неотрицательной и интегрирующейся в $N_{el}$ электронов):

$$ E[\tilde{\rho}] \geq E_{0} $$

### 1.3.2 Уравнения Кона-Шэма

Считать кинетическую энергию мы умеем хорошо только через **орбитали** (как в методе Хартри-Фока). 

Мы можем вернуть орбитали обратно, но не для настоящих электронов, а для **вымышленной системы-двойника**: невзаимодействующих электронов, помещённых в эффективное поле $v_{\text{eff}}(\mathbf{r})$ так, чтобы электронная плотность такой системы совпала с плотностью нашей молекулы.

Тогда энергия нашей системы разбивается на:

$$ E[\rho] = T_{s} + \int v(\mathbf{r})\,\rho(\mathbf{r})\,d\mathbf{r} + J[\rho] + E_{xc}[\rho] $$

* $T_{s}$ - кинетическая энергия невзаимодействующих электронов (точно, через орбитали);
* $\int v\rho$ - притяжение к ядрам (точно);
* $J[\rho]$ - классическое кулоновское отталкивание плотности с самой собой (точно);
* $E_{xc}[\rho]$ - **обменно-корреляционный функционал**: обмен, корреляция и поправка к кинетической энергии.

$E_{xc}$ включает все параметры, которые мы можем оценить только косвенно, а выбор используемых приближений - выбор функционала для расчёта. 

Уравнения Кона-Шэма устроены так же, как уравнения Фока: в базисе они превращаются в то же матричное уравнение $\mathbf{F}\mathbf{C} = \mathbf{S}\mathbf{C}\boldsymbol{\varepsilon}$ и решаются тем же методом SCF. Но за счёт $E_{xc}$ часть электронной корреляции уже включена, в то время как в методе Хартри-Фока её нет совсем. Поэтому DFT даёт результаты значительно лучше метода Хартри-Фока при схожей вычислительной стоимости. 

# 🤖 Глава 2. Компьютерное представление

## 2.1 Базисные функции

И в методе Хартри-Фока-Рутана, и в DFT орбитали ищутся в виде разложения по базису:
$$ \psi_{i}(\mathbf{r}) = \sum_{k=1}^{M} c_{ki}\,\varphi_{k}(\mathbf{r}) $$

Базисная функция, имитирующая атомную орбиталь, может быть разделена на две компоненты - радиальную и угловую.

$$ \phi = R(r)Y({\theta, \varphi}) $$

Угловая часть $Y$ - сферические гармоники, они задают знакомые формы s-, p-, d-орбиталей и одинаковы для любых базисов. 

| Орбиталь | $l$ | $m$ | Угловая часть $Y(\theta,\varphi)$ |
|---|---|---|---|
| $s$ | 0 | $0$ | $\dfrac{1}{\sqrt{4\pi}}$ |
| $p_z$ | 1 | $0$ | $\sqrt{\dfrac{3}{4\pi}}\cos\theta$ |
| $p_x$ | 1 | $+1$ | $\sqrt{\dfrac{3}{4\pi}}\sin\theta\cos\varphi$ |
| $p_y$ | 1 | $-1$ | $\sqrt{\dfrac{3}{4\pi}}\sin\theta\sin\varphi$ |
| $d_{z^2}$ | 2 | $0$ | $\sqrt{\dfrac{5}{16\pi}}\,(3\cos^{2}\theta-1)$ |
| $d_{xz}$ | 2 | $+1$ | $\sqrt{\dfrac{15}{4\pi}}\sin\theta\cos\theta\cos\varphi$ |
| $d_{yz}$ | 2 | $-1$ | $\sqrt{\dfrac{15}{4\pi}}\sin\theta\cos\theta\sin\varphi$ |
| $d_{x^2-y^2}$ | 2 | $+2$ | $\sqrt{\dfrac{15}{16\pi}}\sin^{2}\theta\cos 2\varphi$ |
| $d_{xy}$ | 2 | $-2$ | $\sqrt{\dfrac{15}{16\pi}}\sin^{2}\theta\sin 2\varphi$ |


Вся индивидуальность - в радиальной части $R(r)$. «Естественным» выбором радиальной функции для атомной орбитали является орбиталь типа Слейтера (STO):

$$ R(r) = N r^{n - 1} e^{-\zeta r} $$

При построении матрицы Фока рассчитываются **двухэлектронные интегралы** $J$ и $K$, каждый из которых является произведением пары базисных функций. Например, энергия кулоновского отталкивания ($J$) рассчитывается как:

$$ (\mu\nu|\lambda\sigma) = \iint \varphi_{\mu}(\mathbf{r}_1)\,\varphi_{\nu}(\mathbf{r}_1)\,\frac{1}{r_{12}}\,\varphi_{\lambda}(\mathbf{r}_2)\,\varphi_{\sigma}(\mathbf{r}_2)\,d\mathbf{r}_1\,d\mathbf{r}_2 $$

$$ J_{ij} = \sum_{\mu\nu\lambda\sigma} c_{\mu i}\,c_{\nu i}\,c_{\lambda j}\,c_{\sigma j}\,(\mu\nu|\lambda\sigma) $$


где пара $\varphi_{\mu}\varphi_{\nu}$ образует облако первого электрона, а пара $\varphi_{\lambda}\varphi_{\sigma}$ - второго.

С ними две проблемы:
* Число интегралов растёт как $O(M^4)$, где $M$ - число базисных функций;
* Для STO такие интегралы берутся только численно. 

Мы можем заменить экспоненту $e^{-\zeta r}$ на сумму гауссовых функций (GTO):

$$ R(r) = \sum_i c_i r^{\,l}\, e^{-\alpha_i r^2} $$

Для гауссианов применима **теорема о произведении**, благодаря чему любой многоцентровый интеграл сводится к готовой формуле и считается мгновенно. Для чистых экспонент, $l=0$:

$$ e^{-\alpha\,|\mathbf{r}-\mathbf{A}|^{2}} \cdot e^{-\beta\,|\mathbf{r}-\mathbf{B}|^{2}} = K\,e^{-(\alpha+\beta)\,|\mathbf{r}-\mathbf{P}|^{2}} $$

$$ \mathbf{P} = \frac{\alpha\,\mathbf{A} + \beta\,\mathbf{B}}{\alpha + \beta}, \qquad K = \exp\!\left(-\frac{\alpha\beta}{\alpha+\beta}\,|\mathbf{A}-\mathbf{B}|^{2}\right) $$

In [ ]:
# https://www.basissetexchange.org/
import basis_set_exchange as bse

# Скачиваем STO-3G для водорода в формате ORCA
print(bse.get_basis('STO-3G', elements='H', fmt='orca'))

In [ ]:
import basis_set_exchange as bse

# Экспоненты и коэффициенты STO-3G для водорода — из Basis Set Exchange
shell = bse.get_basis('STO-3G', elements='H')['elements']['1']['electron_shells'][0]
alphas = np.array(shell['exponents'], dtype=float)
coefs  = np.array(shell['coefficients'][0], dtype=float)

r = np.arange(0, 5, 0.01)

# Слейтеровская 1s-орбиталь: N*exp(-ζr); в STO-3G для H принято ζ = 1.24
zeta = 1.24
slater = np.exp(-zeta * r)
slater /= slater.max()

# Контрактация: сумма примитивов d_i * N_i * exp(-α_i r²)
primitives = []
for a, d in zip(alphas, coefs):
    N = (2*a/np.pi)**0.75                  # нормировка s-гауссиана (3/4)
    primitives.append(d * N * np.exp(-a * r**2))
contracted = np.sum(primitives, axis=0)

norm = contracted.max()

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(r, slater, 'k', lw=2, label='STO ($\\zeta$ = 1.24)')
ax.plot(r, contracted / norm, lw=2, label='STO-3G (сумма)')
for i, p in enumerate(primitives, 1):
    ax.plot(r, p / norm, '--', label=f'примитив GTO-{i}')

ax.set_ylim(0, 1.05)
ax.set_xlim(0, 5)
ax.set_title('Слейтеровская орбиталь и её гауссова имитация (H, 1s)')
ax.set_xlabel('r (а.е.)')
ax.set_ylabel('Нормированная амплитуда')
ax.legend()
plt.show()

Базисные наборы разрабатывались «семействами», и каждое имеет единую логику построения и именования. Три наиболее популярных семейства: 
* Базисные наборы Попла: STO-3G, 6-31G(d), 6-31+G(d,p), 6-311++G(2df,2pd)
* Корреляционно согласованные базисные наборы (Даннинга): cc-pVDZ, cc-pVTZ, cc-pVQZ, aug-cc-pVTZ
* Базисные наборы Karlsruhe: def2-SVP, def2-TZVP, def2-QZVP, def2-TZVPD, ma-def2-TZVP

In [ ]:
shells = bse.get_basis('STO-3G', elements='Li', fmt='orca')
print(shells)

# L означает sp-оболочку: 2s- и 2p-функции с ОБЩИМИ экспонентами.
# Поэтому колонок коэффициентов две:
#   первая  - коэффициенты для 2s,
#   вторая  - коэффициенты для 2p.
# Общие экспоненты - экономия базисов Попла.

In [ ]:
# Орбитали лития в минимальном базисе STO-3G
shells = bse.get_basis('STO-3G', elements='Li')['elements']['3']['electron_shells']

r = np.arange(0, 10, 0.01)

def contracted(shell, col, l=0):
    alphas = np.array(shell['exponents'], dtype=float)
    coefs  = np.array(shell['coefficients'][col], dtype=float)
    if l == 0:
        N = (2*alphas/np.pi)**0.75                       # нормировка s-гауссиана
        return sum(d * n * np.exp(-a * r**2) for a, n, d in zip(alphas, N, coefs))
    else:
        N = (2*alphas/np.pi)**0.75 * 2*np.sqrt(alphas)   # нормировка p-гауссиана
        return sum(d * n * r * np.exp(-a * r**2) for a, n, d in zip(alphas, N, coefs))

core_1s = contracted(shells[0], 0)                 # блок S: 1s
val_2s  = contracted(shells[1], 0)                 # блок L, 1-й столбец: 2s
val_2p  = contracted(shells[1], 1, l=1)            # блок L, 2-й столбец: 2p

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.axhline(0, color='gray', lw=0.8)
ax.plot(r, core_1s / core_1s.max(), 'k', lw=2, label='остовная 1s (S 3)')
ax.plot(r, val_2s / np.abs(val_2s).max(), lw=2.5, label='валентная 2s (L 3, столбец s)')
ax.plot(r, val_2p / np.abs(val_2p).max(), lw=2.5, label='валентная 2p (L 3, столбец p)')

ax.set_ylim(0, 1.05)
ax.set_xlim(0, 10)
ax.set_title('Орбитали лития в STO-3G: 1s, 2s и 2p')
ax.set_xlabel('r (а.е.)')
ax.set_ylabel('Нормированная амплитуда')
ax.legend()
plt.show()


#### 2.1.1 Валентные орбитали

В минимальном базисе (STO-3G) на каждую атомную орбиталь приходится ровно **одна** функция фиксированной формы. Но орбитали атомов неравноценны по "их вкладу в химию":

* **остовные орбитали** (например, 1s углерода) лежат глубоко по энергии и практически не участвуют в химии элемента. Их оставляем одной жёсткой функцией: несколько примитивов. 

* **валентные орбитали** отвечают за взаимодействие с орбиталями других атомов, участвуют в формировании связей. Им можно выдать по две и более **независимых** функций разной ширины: компактные «внутренние» (большие экспоненты $\alpha$) и рыхлые «внешние» (малое значение $\alpha$):

$$ \psi_{\text{вал}}(r) \;=\; c_{1}\,\underbrace{e^{-\alpha_{1} r^{2}}}_{\text{компактная}} \;+\; c_{2}\,\underbrace{e^{-\alpha_{2} r^{2}}}_{\text{рыхлая}}, \qquad \alpha_{1} \gg \alpha_{2} $$

Число независимых функций на валентную орбиталь определяют в базисных наборах по букве $\zeta$: double-zeta (DZ) — две, triple-zeta (TZ) — три, quadruple-zeta (QZ) — четыре.

In [ ]:
shells = bse.get_basis('3-21G', elements='H', fmt='orca')
print(shells)

In [ ]:
# Пластичность валентной 2s: минимальный базис (STO-3G) против double-zeta (3-21G)
r = np.arange(0, 14, 0.01)

def contracted(shell, col):
    alphas = np.array(shell['exponents'], dtype=float)
    coefs  = np.array(shell['coefficients'][col], dtype=float)
    N = (2*alphas/np.pi)**0.75
    return sum(d * n * np.exp(-a * r**2) for a, n, d in zip(alphas, N, coefs))

def normed(f):   
    return f / np.sqrt(np.sum(f**2 * 4*np.pi*r**2) * (r[1]-r[0]))

sto  = bse.get_basis('STO-3G', elements='Li')['elements']['3']['electron_shells']
p321 = bse.get_basis('3-21G',  elements='Li')['elements']['3']['electron_shells']

sto_2s = contracted(sto[1], 0)                      # единственная 2s в STO-3G
inner  = normed(contracted(p321[1], 0))             # 3-21G: блок L 2
outer  = normed(contracted(p321[2], 0))             # 3-21G: блок L 1

def mix(w):                                         # смесь
    psi = (1 - w) * inner + w * outer
    return psi / psi.max()

cmap = plt.cm.bwr                                 # палитра ленты
c_lo, c_hi = 0.0, 0.95                             # рабочий диапазон палитры

fig, ax = plt.subplots(figsize=(12, 4.5))

# градиентная лента: заливка между каждой парой соседних смесей
ws = np.linspace(0, 0.95, 41)
curves = [mix(w) for w in ws]
band_colors = cmap(np.linspace(c_lo, c_hi, len(ws) - 1))
for lo, hi, c in zip(curves[:-1], curves[1:], band_colors):
    ax.fill_between(r, lo, hi, color=c, alpha=0.5, lw=0)

# семь опорных кривых поверх ленты: крайние жирно, промежуточные тоньше
ws_lines = np.linspace(0, 0.95, 7)
line_colors = cmap(np.linspace(c_lo, c_hi, 7))
for w, c in zip(ws_lines, line_colors):
    edge = w in (ws_lines[0], ws_lines[-1])
    ax.plot(r, mix(w), color=c, lw=2.5 if edge else 1.3, label=f'3-21G: доля рыхлой = {w:.2f}')

ax.plot(r, sto_2s / sto_2s.max(), lw=2.5, ls='--', label='STO-3G: единственная форма', color="black")

ax.plot(r, (inner / inner.max())/2, color='#F46036', ls='--', lw=2, label='3-21G: компактная функция (блок L 2)')
ax.plot(r, (outer / outer.max())/4, color='#6AB547', ls='--', lw=2, label='3-21G: рыхлая функция (блок L 1)')

ax.set_xlim(0, 12)
ax.set_ylim(0, 1.05)
ax.set_xlabel('r (а.е.)')
ax.set_title('Пластичность валентной 2s орбитали лития: STO-3G против 3-21G')
ax.set_ylabel('Нормированная амплитуда')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


### 2.1.2 Поляризационные функции

Расщепление валентности дало орбиталям свободу **по размеру**, но не **по форме**: сколько бы s-функций мы ни выдали водороду, его облако остаётся сферическим. А атом в молекуле несимметричен - его плотность смещена к соседнему атому при образовании химической связи.

Отсюда правило: **поляризационные функции — это функции с $l$ на единицу выше, чем у занятых валентных орбиталей**:

* для H (валентная s) — добавляют p-функции;
* для B–F (валентные s, p) — добавляют d-функции;
* для переходных металлов — f-функции.

В именах базисов: 6-31G(d) - d-функции на тяжёлых атомах. 6-31G(d,p) - ещё и p на водородах (старая запись: 6-31G\* и 6-31G\*\*). В современных семействах поляризация встроена по умолчанию: def2-SVP, cc-pVnZ (*p = polarized*).

Поляризационные функции критичны там, где важна форма орбитали: валентные углы (без d-функций на азоте пирамида NH₃ стремится стать плоской), гипервалентные соединения, водородные связи.

In [ ]:
shells = bse.get_basis('6-31G(d,p)', elements='H', fmt='orca')
print(shells)

In [ ]:
# Поляризация: смесь s- и p-функций базиса 6-31G(d,p) для водорода
z = np.arange(-3, 3, 0.01)

shells = bse.get_basis('6-31G(d,p)', elements='H')['elements']['1']['electron_shells']

# s-функция: контрактация из 3 примитивов (блок S 3)
alphas = np.array(shells[0]['exponents'], dtype=float)
coefs  = np.array(shells[0]['coefficients'][0], dtype=float)
N = (2*alphas/np.pi)**0.75
s = sum(d * n * np.exp(-a * z**2) for a, n, d in zip(alphas, N, coefs))
s /= s.max()

# p-функция: поляризационная добавка (блок P 1, α = 1.1)
a_p = float(shells[2]['exponents'][0])
p = (2*a_p/np.pi)**0.75 * 2*np.sqrt(a_p) * z * np.exp(-a_p * z**2)
p /= np.abs(p).max()

def mix(w):                                         # смесь: (1-w)·s + w·p
    psi = (1 - w) * s + w * p
    return psi / psi.max()

cmap = plt.cm.bwr                                   # палитра ленты
c_lo, c_hi = 0.0, 0.95                              # рабочий диапазон палитры

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.axhline(0, color='gray', lw=0.8)
ax.axvline(0, color='gray', lw=0.8)

# градиентная лента: заливка между каждой парой соседних смесей
ws = np.linspace(0, 0.35, 41)
curves = [mix(w) for w in ws]
band_colors = cmap(np.linspace(c_lo, c_hi, len(ws) - 1))
for lo, hi, c in zip(curves[:-1], curves[1:], band_colors):
    ax.fill_between(z, lo, hi, color=c, alpha=0.5, lw=0)

# семь опорных кривых поверх ленты: крайние жирно, промежуточные тоньше
ws_lines = np.linspace(0, 0.35, 7)
line_colors = cmap(np.linspace(c_lo, c_hi, 7))
for w, c in zip(ws_lines, line_colors):
    edge = w in (ws_lines[0], ws_lines[-1])
    ax.plot(z, mix(w), color=c, lw=2.5 if edge else 1.3, label=f'доля p = {w:.2f}')

ax.set_xlim(-3, 3)
ax.set_ylim(-0.25, 1.05)
ax.set_xlabel('z (а.е.)')
ax.set_title('Поляризация: смеси s- и p-функций водорода в 6-31G(d,p)')
ax.set_ylabel('Нормированная амплитуда')
ax.legend()
plt.tight_layout()
plt.show()


### 2.1.3 Диффузные функции

Все функции, которые мы видели до сих пор, локализованы в пределах нескольких ангстрем от ядра. Но существуют системы, где электронная плотность может быть удалена от ядра дальше:

* **анионы** - дополнительный электрон связан слабее, и его облако молекулярной орбитали отдалено;
* **возбуждённые (ридберговские) состояния**;
* **слабые взаимодействия** - водородные связи, ван-дер-ваальсовы взаимодействия;

Можно добавить в базис дополнительные **диффузные функции** - обычные гауссианы, но с очень маленькой экспонентой $\alpha \approx 0.01\!-\!0.1$. Ведь чем меньше $\alpha$, тем шире функция. 

В базисных наборах Попла диффузные функции добавляются как `+`, `++` (6-31+G, 6-311++G). В базисных наборах Даннинга - `aug-` (aug-cc-pVTZ). Для базисных наборов Karlsruhe - `D` или `ma-` (def2-TZVPD, ma-def2-TZVP).

Диффузные функции с соседних атомов сильно перекрываются друг с другом, базис становится почти **линейно зависимым**, и SCF сходится хуже и медленнее.

In [ ]:
# Диффузная функция в файле базиса: 6-31++G для водорода
print(bse.get_basis('6-31++G', elements='H', fmt='orca'))

# Три блока S вместо двух знакомых по 6-31G:
#   S 3 — компактная (контрактация, α = 18.7 / 2.83 / 0.64);
#   S 1 — рыхлая валентная (α = 0.16);
#   S 1 — диффузная

In [ ]:
r = np.arange(0, 20, 0.01)
dr = r[1] - r[0]

shells = bse.get_basis('6-31++G', elements='H')['elements']['1']['electron_shells']

def contracted(shell):
    alphas = np.array(shell['exponents'], dtype=float)
    coefs  = np.array(shell['coefficients'][0], dtype=float)
    N = (2*alphas/np.pi)**0.75
    return sum(d * n * np.exp(-a * r**2) for a, n, d in zip(alphas, N, coefs))

def rho(f):
    d = f**2 * 4*np.pi*r**2
    return d / (np.sum(d) * dr)

names  = ['(S 3 α = 18.7-0.64)', ' (S 1, α = 0.16)', 'диффузная (S 1, α = 0.036)']
colors = ['red', 'blue', 'green']

fig, ax = plt.subplots(figsize=(12, 4.5))
for sh, name, c in zip(shells, names, colors):
    ax.plot(r, rho(contracted(sh)), color=c, lw=2.5, label=f'{name}')

ax.set_xlim(0, 12)
ax.set_ylim(0, None)
ax.set_title('Диффузная функция для атома H, 6-31++G')
ax.set_xlabel('r (а.е.)')
ax.set_ylabel('Нормированная радиальная плотность')
ax.legend()
plt.tight_layout()
plt.show()


## 2.2 Методы пост-Хартри-Фока

#### 2.2.1 Проблема электронной корреляции

Главное упрощение метода Хартри-Фока: каждый электрон движется в **усреднённом** поле остальных частиц и получаемая энергия оказывается больше реальной. Разницу между точной энергией и энергией, полученной методом Хартри-Фока в данном базисном наборе, называют **энергией электронной корреляции**. 

$$ E_{\text{corr}} = E_{\text{точн}} - E_{\text{ХФ}} < 0 $$

Ошибка в методе Хартри-Фока систематическая и при расчёте относительной энергии частично сокращается - поэтому метод качественно работает, но для количественных ответов корреляцию нужно учитывать.

Корреляцию принято делить на два типа:

* **динамическая** — обусловлена коррелированным движением электронов, возникающим из-за их кулоновского отталкивания.

* **статическая** — возникает, когда у системы **несколько электронных конфигураций, близких по энергии**, и один детерминант в принципе плохое приближение. Возникает, например, при увеличении длины или разрыве химической связи, в бирадикальных системах и комплексах переходных металлов.

Методы, которые стартуют с решения ХФ и позволяют уточнить корреляционную энергию, называют **пост-хартри-фоковскими**.

### 2.2.2 Метод возмущений Мёллера-Плессета

Первый способ рассчитать поправку для электронной корреляции - применить **теорию возмущений** для решённой задачи Хартри-Фока. Нулевой и первый порядки такой теории в сумме дают просто $E_{\text{ХФ}}$, так что первая настоящая поправка появляется во **втором порядке** теории возмущений:


$$ E_{\text{MP2}} = E_{\text{ХФ}} + \sum_{i<j}^{\text{зан}} \sum_{a<b}^{\text{вирт}} \frac{\big|\langle ij \| ab \rangle\big|^{2}}{\varepsilon_i + \varepsilon_j - \varepsilon_a - \varepsilon_b} $$

Мы берём каждую пару занятых орбиталей $i, j$ и пробуем скомбинировать их электроны по паре виртуальных $a, b$ - это **двукратное возбуждение**. Все слагаемые отрицательны (знаменатель $< 0$): каждая пара виртуальных орбиталей снижает энергию системы.

Свойства метода MP2:
* возвращает **~80–90% корреляционной энергии** — за одну формулу без итераций поверх готового SCF;
* масштабируется как $O(M^5)$ (против $O(M^4)$ у метода ХФ) от числа базисных функций $(M)$;

На практике имеет смысл применять только низкие порядки теории возмущений.

### 2.2.3 Метод конфигурационного взаимодействия (CI)

Метод Хартри-Фока определяет энергетически оптимальную пробную волновую функцию с одним детерминантом (в пределах заданного базисного набора). Поэтому очевидно, что для улучшения результатов отправной точкой может служить пробная волновая функция, содержащая более одного детерминанта Слейтера. Общую волновую функцию, составленную из нескольких детерминантов Слейтера, можно записать в виде: 

$$ \Psi_{\text{CI}} = a_{0}\,\Psi_{\text{ХФ}} + \sum_{S} a_{S}\,\Psi_{S} + \sum_{D} a_{D}\,\Psi_{D} + \ldots $$

Коэффициенты $a$ также находятся вариационно. Если включить все возможные детерминанты в заданном базисе,
то вся электронная корреляция (в данном базисе) восстанавливается - **полное CI (FCI)**. В случае бесконечного базиса уравнение Шредингера для молекулярной системы методом FCI решается точно.

Несмотря на недостижимость FCI в реальных расчётах, сама идея может быть использована в усечённом виде, например с использованием только одно- (CIS) и двукратно возбуждённых детерминантов  **CISD** (при вычислительной стоимости $O(M^6)$). Хотя на практике усечённые методы CI вытеснены другими методами, например **методом связанных кластеров**

### 2.2.4 Метод связанных кластеров (Coupled Cluster)

CI складывал детерминанты **суммой**. Метод связанных кластеров делает это **экспонентой**:

$$ \Psi_{\text{CC}} = e^{\hat{T}}\,\Psi_{\text{ХФ}}, \qquad \hat{T} = \hat{T}_1 + \hat{T}_2 + \hat{T}_3 + \ldots $$

где $\hat{T}_1$ генерирует все однократные возбуждения, $\hat{T}_2$ - все двукратные, и так далее. При раскрытии $e^{\hat{T}}$ мы получаем: 
 
$$ e^{\hat{T}} = 1 + \hat{T} + \tfrac{1}{2}\hat{T}^{2} + \tfrac{1}{6}\hat{T}^{3} + \ldots = 1 + \hat{T}_1 + \big(\hat{T}_2 + \tfrac{1}{2}\hat{T}_1^{2}\big) + \big(\hat{T}_3 + \hat{T}_2\hat{T}_1 + \tfrac{1}{6}\hat{T}_1^{3}\big)  + \ldots $$

Первое слагаемое формирует опорное состояние Хартри-Фока, а второе - все однократно возбужденные состояния, а первая скобка формирует все двукратно возбужденные состояния, которые можно рассматривать как связанные ($T2$) или несвязанные ($T1^2$). Дальше по тому же принципу: каждая скобка собирает возбуждения одной кратности из связанных кластеров и всех их независимых комбинаций. Отсюда и название метода - **связанных кластеров**.

Остановимся на двукратном $\hat{T}$: $\hat{T} = \hat{T}_1 + \hat{T}_2$ - это метод **CCSD**. Связанные $\hat{T}_3$ и $\hat{T}_4$ исчезают, но скобки не пустеют: трёхкратные возбуждения выживают в несвязанном виде $\hat{T}_2\hat{T}_1$, а четырёхкратные — в виде $\tfrac{1}{2}\hat{T}_2^{2}$: «двукратное возбуждение в одном месте и одновременно двукратное в другом», то есть четырёхкратные возбуждения, собранные как **произведения** двукратных.

Лесенка метода:
* **CCSD** ($\hat{T}_1 + \hat{T}_2$): масштабирование $O(M^6)$
* **CCSDT** ($\hat{T}_1 + \hat{T}_2 +  \hat{T}_3$): масштабирование $O(M^8)$
* **CCSD(T)**: трёхкратные возбуждения добавлены не полностью, а оценены по теории возмущений - отсюда скобки в названии. Масштабирование $O(M^7)$. CCSD(T) **«золотой стандарт» квантовой химии**: в хорошем базисе даёт точность в оценке энергии (~1 ккал/моль).

## 2.3 Методы теории функционала плотности

В уравнениях Кона-Шэма энергия системы может быть рассчитана по уравнению:

$$ E[\rho] = T_{s} + \int v(\mathbf{r})\,\rho(\mathbf{r})\,d\mathbf{r} + J[\rho] + E_{xc}[\rho] $$

Все слагаемые рассчитываются точно, кроме одного - обменно-корреляционного функционала $E_{xc}[\rho]$. Теорема Хоэнберга-Кона гарантирует, что точный функционал *существует*, но не говорит, как он выглядит. 

Для систематизации приближений, используемых в обменно-корреляционных функционалах, Джон Пердью предложил классифицировать функционалы по тому, как они используют электронную плотность для вычисления $E_{xc}[\rho]$. Такая классификация позже стала называться «лестницей Якоба-Пердью»:

1. **LDA**: $\varepsilon_{xc}(\rho)$ - учитывает только локальную электронную плотность в точке. Обмен и корреляция берутся как у **однородного электронного газа** той же плотности. Сложность $O(M^3)$.

2. **GGA**: $\varepsilon_{xc}(\rho,\ \nabla\rho)$ - учитывается также и градиент электронной плотности (PBE, BLYP). Сложность $O(M^3)$.

3. **meta-GGA**: $\varepsilon_{xc}(\rho,\ \nabla\rho,\ \tau)$, - дополнительно учитывает плотность кинетической энергии. $\tau = \tfrac{1}{2}\sum_{i}^{\text{зан}} |\nabla\psi_i|^2$  (TPSS, SCAN). Сложность $O(M^3)$.

4. **Hybrid** - учитывает занятые орбитали целиком и к DFT-обмену подмешивается доля $a$ *точного* обмена Хартри-Фока:
$$ E_{xc}^{\text{гибрид}} = a\,E_{x}^{\text{ХФ}} + (1-a)\,E_{x}^{\text{DFT}} + E_{c}^{\text{DFT}} $$
(B3LYP: $a = 0.20$, PBE0: $a = 0.25$). Сложность $O(M^4)$.

5. **Double hybrid** - дополнительно учитываются виртуальные орбитали и к энергии добавляется доля $b$ MP2-подобной корреляции:
$$ E_{xc}^{\text{дв.гибрид}} = a\,E_{x}^{\text{ХФ}} + (1-a)\,E_{x}^{\text{DFT}} + (1-b)\,E_{c}^{\text{DFT}} + b\,E_{c}^{\text{MP2}} $$
(B2PLYP: $a = 0.53$, $b = 0.27$). Сложность $O(M^5)$.


# 🚀 Глава 3. Дополнительные материалы

### 3.1 Приближение "Resolution of the identity (RI)"

В методе Хартри-Фока вычислительная сложность $O(M^4)$ определяется расчётом **двухэлектронных интегралов** $J$ и $K$ : 

$$ (\mu\nu|\lambda\sigma) = \iint \varphi_{\mu}(\mathbf{r}_1)\,\varphi_{\nu}(\mathbf{r}_1)\,\frac{1}{r_{12}}\,\varphi_{\lambda}(\mathbf{r}_2)\,\varphi_{\sigma}(\mathbf{r}_2)\,d\mathbf{r}_1\,d\mathbf{r}_2 $$


Приближение RI позволяет значительно удешевить расчёт таких интегралов почти без потери точности, используя разложение по одному **вспомогательному базису** $\{P\}$:

$$ \varphi_{\mu}(\mathbf{r})\,\varphi_{\nu}(\mathbf{r}) \;\approx\; \sum_{P} c_{\mu\nu}^{P}\, P(\mathbf{r}) $$

Тогда четырёхиндексный интеграл распадается на комбинацию трёх- и двухиндексных:

$$ (\mu\nu|\lambda\sigma) \;\approx\; \sum_{P,Q} (\mu\nu|P)\,\big[\mathbf{V}^{-1}\big]_{PQ}\,(Q|\lambda\sigma), \qquad V_{PQ} = (P|Q) $$

Практический словарик:

* **RI-J** — RI для кулоновской части DFT. 
* **RIJCOSX** — для гибридных функционалов: кулоновская часть считается через RI, а точный обмен полуаналитическим методом COSX.
* **RI-MP2** — RI для интегралов MP2

Вспомогательные базисы **/J** (для RI-J) и **/C** (для RI-MP2 и двойных гибридов). 


### 3.2 Дисперсионные взаимодействия

У полулокальных функционалов (LDA, GGA, meta-GGA) есть проблема с оценкой нековалентных взаимодействий на расстояниях Ван-дер-Ваальса. Один из способов это исправить - использовать эмпирическую поправку Гримме **DFT-D** к энергии DFT:

$$ E_{\text{disp}} = -\sum_{A>B} \left( \frac{C_6^{AB}}{R_{AB}^{6}} + \frac{C_8^{AB}}{R_{AB}^{8}} \right) f_{\text{damp}}(R_{AB}) $$

### 3.3 Релятивистские поправки

Ранее мы использовали нерелятивистское уравнение Шрёдингера и для лёгких ядер (до ядер Fe) это приближение работает отлично. Но скорость ближайших к ядру электронов растёт с зарядом ядра, в результате чего происходит сжатие близких к ядру электронных оболочек и заряд ядра экранируется, а внешние d- и f-оболочки в ответ расширяются.

Один из эффективных способов учёта релятивистских эффектов в квантово-химических расчётах - использование релятивистских поправок:

* **ECP (эффективные остовные потенциалы)** — самый дешёвый путь, при котором остов тяжёлого атома заменяется параметризованным потенциалом. 

* **скалярно-релятивистские гамильтонианы** — поправки **ZORA** (дешевле) и **DKH** (дороже) приближённо сводят уравнение Дирака к привычному однокомпонентному виду. При этом все электроны остаются в расчёте, а релятивистские эффекты учитываются через поправку к кинетической энергии. Для таких расчётов обычно используют специальные базисные наборы, например ZORA-def2-TZVP.

### 3.4 Экстраполяция базисного набора

Ошибка, связанная с неполнотой базисного набора, уменьшается с его расширением, однако вычислительная стоимость при этом может возрастать в разы. Используя результаты расчётов в двух базисах, можно оценить предел бесконечно полного базиса (complete basis set, CBS).

Посчитав корреляционную энергию в двух соседних базисах с кардинальными номерами $X$ и $Y$ (например, TZ: $X=3$ и QZ: $Y=4$), предел может быть найден алгебраически:

$$ E_{\text{CBS}}^{\text{corr}} = \frac{Y^{3} E_{Y}^{\text{corr}} - X^{3} E_{X}^{\text{corr}}}{Y^{3} - X^{3}} $$


### 3.5 Континуальные модели растворителя

До сих пор система рассматривалась в вакууме при 0 K. Однако в растворе молекула взаимодействует с окружающей средой, что может стабилизировать отдельные конформации и полярные структуры, смещать равновесия реакций и изменять спектральные свойства. Один из способов учесть среду молекулы в квантовой химии - использовать **Континуальные (неявные) модели**, представляющие растворитель как бесструктурную поляризуемую среду с диэлектрической проницаемостью $\varepsilon$.

Механика у всех моделей общая:

1. в среде вырезается **полость** по форме молекулы;
2. электронная плотность молекулы поляризует среду и на поверхности полости наводятся заряды;
3. эти заряды создают **реактивное поле**, которое действует обратно на молекулу;
4. пункты 2-3 согласуются итерационно, прямо внутри SCF-цикла.

| Модель | Идея |
|---|---|
| **CPCM** | среда приближается идеальным проводником — уравнения сильно упрощаются | 
| **IEF-PCM** | честное решение электростатики диэлектрика |
| **SMD** | PCM + параметризованные неэлектростатические вклады|

#  🐋 Глава 4. Основы работы с пакетом Orca

Мануал для ORCA: https://orca-manual.mpi-muelheim.mpg.de/

Туториалы для ORCA: https://www.faccts.de/docs/orca/6.0/tutorials/

In [ ]:
import os

ORCA = "/opt/orca/orca" # ORCA запускается только по абсолютному пути
print("ORCA найдена:", os.path.isfile(ORCA))

In [ ]:
# Рабочая папка для расчётов
os.chdir(os.path.expanduser("/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_4/"))

# Входной файл
input_file = """\
! HF STO-3G          # Основные ключи, порядок не важен
%maxcore 2000        # Дополнительные настройки
%pal nprocs 8 end    # Расчёт на 8 ядрах

* xyz 0 1
O   0.000000   0.000000   0.117300
H   0.000000   0.757200  -0.469200
H   0.000000  -0.757200  -0.469200
* """ 

with open("water.inp", "w") as f:
    f.write(input_file)

In [ ]:
# Инициализация библиотек ORCA
os.environ["PATH"] = "/opt/orca:/opt/openmpi/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = "/opt/orca:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["OMPI_MCA_plm"] = "isolated"

In [ ]:
# Запуск расчёта
!/opt/orca/orca water.inp > water.out --oversubscribe 
!tail -n 2 water.out

#### 4.1 Полезные ключевые слова

**Тип задачи**

| Ключевое слово | Что делает |
|---|---|
| `SP` | энергия в одной точке; считается по умолчанию, если тип задачи не указан |
| `Opt` | оптимизация геометрии  |
| `Freq`, `NumFreq` | частоты колебаний, аналитически и численно |

**Метод**

| Раздел | Ключевые слова | Комментарий |
|---|---|---|
| 1.2 Хартри-Фок | `HF`, `UHF`, `ROHF` | для открытой оболочки `HF` сам переключается на UHF |
| 2.2 Теория возмущений | `MP2`, `RI-MP2`, `SCS-MP2` | `RI-MP2` требует вспомогательный базис `/C` |
| 2.2 Связанные кластеры | `CCSD`, `CCSD(T)`, `DLPNO-CCSD(T)` | `DLPNO-` - приближённый вариант для больших молекул, тоже требует `/C` |
| 2.3 LDA и GGA | `LDA`, `PBE`, `BLYP`, `BP86` | |
| 2.3 meta-GGA | `TPSS`, `r2SCAN` | |
| 2.3 Гибридные | `B3LYP`, `PBE0`, `TPSSh`, `M062X` | |
| 2.3 С разделением по дальности | `CAM-B3LYP`, `wB97X-V`, `wB97X-D4` | |
| 2.3 Двойные гибриды | `B2PLYP` | требует вспомогательный базис `/C` |
| Композитные методы | `HF-3c`, `B97-3c`, `r2SCAN-3c`, `PBEh-3c` | базис и поправки уже внутри, базис указывать не нужно |

**Базисный набор (раздел 2.1)**

| Семейство | Ключевые слова |
|---|---|
| Попла | `STO-3G`, `3-21G`, `6-31G(d)`, `6-31G(d,p)`, `6-31+G(d)`, `6-311++G(d,p)` |
| Даннинга | `cc-pVDZ`, `cc-pVTZ`, `cc-pVQZ`, `aug-cc-pVDZ`, `aug-cc-pVTZ` |
| Karlsruhe | `def2-SVP`, `def2-TZVP`, `def2-TZVPP`, `def2-QZVP`, `def2-TZVPD`, `ma-def2-TZVP` |

Старая запись тоже работает: `6-31G*` - это `6-31G(d)`, а `6-31G**` - это `6-31G(d,p)`.

**Поправки и приближения (глава 3)**

| Тема | Ключевые слова | Комментарий |
|---|---|---|
| Приближение RI | `RIJCOSX`, `def2/J`, `def2-TZVP/C`, `AutoAux`, `NoRI` | в DFT включено по умолчанию, базис `def2/J` ORCA подставляет сама |
| Дисперсия | `D3BJ`, `D4` | добавляется к функционалу: `! B3LYP D3BJ def2-TZVP` |
| Растворитель | `CPCM(water)`, `CPCM(ethanol)` | название растворителя в скобках |
| Релятивистские эффекты | `ZORA`, `DKH` | вместе со своим базисом: `ZORA-def2-TZVP`, `DKH-def2-TZVP` и `SARC/J` |
| Экстраполяция базиса | `Extrapolate(2/3,cc)` | два расчёта в cc-pVDZ и cc-pVTZ и предел полного базиса |

**Сходимость SCF (раздел 1.2.1)**

| Ключевое слово | Что делает |
|---|---|
| `TightSCF`, `VeryTightSCF` | более жёсткие пороги сходимости |
| `SlowConv`, `VerySlowConv` | сильное демпфирование, если SCF «раскачивается» и не сходится |
| `%scf MaxIter 300 end` | больше итераций (по умолчанию 125) |
| `MORead` и `%moinp "old.gbw"` | взять стартовые орбитали из предыдущего расчёта |
| `DefGrid3` | более плотная сетка интегрирования в DFT |

**Ресурсы и координаты**

| Строка | Что делает |
|---|---|
| `%pal nprocs 4 end` | расчёт на 4 ядрах; ORCA при этом нужно запускать по полному пути |
| `%maxcore 2000` | память на одно ядро в МБ |
| `* xyz 0 1` ... `*` | координаты во входном файле: заряд 0, мультиплетность 1 |
| `* xyzfile 0 1 water.xyz` | координаты из отдельного xyz-файла |

**Типичные сочетания**

```
! r2SCAN-3c Opt Freq                          # быстрая оптимизация геометрии и частоты
! B3LYP D3BJ def2-TZVP TightSCF               # стандартный расчёт энергии методом DFT
! PBE0 D4 def2-TZVP CPCM(water)               # то же в растворителе
! DLPNO-CCSD(T) cc-pVTZ cc-pVTZ/C TightSCF    # точная энергия в готовой геометрии
```


# 📚 Дополнительная литература 

1. **F. Jensen. Introduction to Computational Chemistry** (3rd ed., Wiley, 2017)

2. **C. J. Cramer. Essentials of Computational Chemistry: Theories and Models** (2nd ed., Wiley, 2004) 

3. **В. Г. Цирельсон. Квантовая химия. Молекулы, молекулярные системы и твёрдые тела** (М.: БИНОМ, 2010)
